# Đánh giá chatbot RAG pháp luật (3 mức)

Notebook này đánh giá theo 3 mức chính:

1. **Retrieval**: Hệ thống có tìm đúng điều luật kỳ vọng không?
2. **Generation**: Câu trả lời của LLM có đúng nội dung kỳ vọng không?
3. **End-to-End**: Câu trả lời cuối vừa đúng nội dung vừa bám đúng căn cứ pháp lý không?

> Dữ liệu đánh giá: `scripts/eval_questions.csv`
> 
> Vector DB: `vector_store/index.faiss` + `vector_store/metadata.json`

In [36]:
import json
import os
import re
import unicodedata
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer

load_dotenv()

# Paths
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
INDEX_PATH = PROJECT_ROOT / "vector_store" / "index.faiss"
METADATA_PATH = PROJECT_ROOT / "vector_store" / "metadata.json"
CORPUS_PATH = PROJECT_ROOT / "processed" / "legal_corpus.csv"
EVAL_PATH = NOTEBOOK_DIR / "eval_questions.csv"

# Config
TOP_K = 5
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4o-mini")
OPEN_ROUTER_API_KEY = os.getenv("OPEN_ROUTER_API_KEY")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EVAL_PATH exists:", EVAL_PATH.exists())
print("INDEX_PATH exists:", INDEX_PATH.exists())
print("METADATA_PATH exists:", METADATA_PATH.exists())

PROJECT_ROOT: d:\sourcecode\4th year\2nd term\CĐHTTT\BTLCuoiKy
EVAL_PATH exists: True
INDEX_PATH exists: True
METADATA_PATH exists: True


In [37]:
def normalize_text(text: str) -> str:
    text = str(text).lower().strip()
    text = unicodedata.normalize("NFD", text)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [38]:
def extract_expected_signature(expected_article: str) -> dict:
    raw = str(expected_article)
    norm = normalize_text(raw)

    article_no = None
    year = None
    doc_type = None
    doc_no = None

    m_article = re.search(r"dieu\s*(\d+)", norm)
    if m_article:
        article_no = m_article.group(1)

    m_year = re.search(r"(19|20)\d{2}", norm)
    if m_year:
        year = m_year.group(0)

    if "nghi dinh" in norm:
        doc_type = "ND"
        m_doc_no = re.search(r"nghi dinh\s*(\d+)", norm)
        if m_doc_no:
            doc_no = m_doc_no.group(1)
    elif "bo luat" in norm:
        doc_type = "BOLUAT"
    elif "luat" in norm:
        doc_type = "LUAT"

    return {
        "article_no": article_no,
        "year": year,
        "doc_type": doc_type,
        "doc_no": doc_no,
        "raw": raw,
        "norm": norm,
    }

In [39]:
def parse_id_signature(doc_id: str) -> dict:
    parts = str(doc_id).split("|")[0].split("_")
    doc_type = parts[0] if len(parts) > 0 else None
    doc_no = parts[1] if len(parts) > 1 and parts[1].isdigit() else None
    year = parts[2] if len(parts) > 2 and parts[2].isdigit() else None
    return {"doc_type": doc_type, "doc_no": doc_no, "year": year}

In [40]:
def load_faiss_index(path: Path) -> faiss.Index:
    resolved = path.resolve()
    if not resolved.exists():
        raise FileNotFoundError(f"Không tìm thấy index: {resolved}")
    try:
        return faiss.read_index(str(resolved))
    except RuntimeError:
        raw = resolved.read_bytes()
        return faiss.deserialize_index(np.frombuffer(raw, dtype=np.uint8))

In [41]:
print("Loading index/metadata...")
index = load_faiss_index(INDEX_PATH)

with METADATA_PATH.open("r", encoding="utf-8") as f:
    payload = json.load(f)

if isinstance(payload, dict):
    metadata = payload.get("records", [])
    embedding_model_name = payload.get("embedding_model", "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    metric = payload.get("metric", "l2")
else:
    metadata = payload
    embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    metric = "l2"

if metadata and "content" not in metadata[0]:
    corpus = pd.read_csv(CORPUS_PATH)
    id_to_content = dict(zip(corpus["id"].astype(str), corpus["content"].fillna("").astype(str)))
    for row in metadata:
        row["content"] = id_to_content.get(str(row.get("id", "")), "")

print("Loading embedding model:", embedding_model_name)
embedder = SentenceTransformer(embedding_model_name)

print(f"Loaded {len(metadata)} metadata records; index dim={index.d}; metric={metric}")

Loading index/metadata...
Loading embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Loaded 8950 metadata records; index dim=384; metric=cosine


In [42]:
def retrieve(question: str, top_k: int = TOP_K):
    query_vec = embedder.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=(metric == "cosine"),
    ).astype("float32")

    if query_vec.shape[1] != index.d:
        raise ValueError(f"Lệch chiều vector: {query_vec.shape[1]} vs {index.d}")

    distances, idxs = index.search(query_vec, top_k)
    hits = []
    for rank, idx in enumerate(idxs[0], start=1):
        if idx < 0 or idx >= len(metadata):
            continue
        row = metadata[idx]
        hits.append({
            "rank": rank,
            "distance": float(distances[0][rank - 1]),
            "id": str(row.get("id", "")),
            "doc_type": str(row.get("doc_type", "")),
            "year": str(row.get("year", "")),
            "article_no": str(row.get("article_no", "")),
            "article_title": str(row.get("article_title", "")),
            "content": str(row.get("content", "")),
        })
    return hits

In [43]:
def is_expected_hit(expected_sig: dict, hit: dict) -> bool:
    if expected_sig["article_no"] and str(hit.get("article_no", "")) != expected_sig["article_no"]:
        return False

    if expected_sig["year"] and str(hit.get("year", "")) != expected_sig["year"]:
        return False

    if expected_sig["doc_type"] and str(hit.get("doc_type", "")) != expected_sig["doc_type"]:
        return False

    if expected_sig["doc_type"] == "ND" and expected_sig["doc_no"]:
        id_sig = parse_id_signature(hit.get("id", ""))
        if str(id_sig.get("doc_no", "")) != expected_sig["doc_no"]:
            return False

    return True

In [44]:
def first_correct_rank(expected_sig: dict, hits: list[dict]):
    for h in hits:
        if is_expected_hit(expected_sig, h):
            return h["rank"]
    return None

In [45]:
def token_f1(pred: str, ref: str) -> float:
    p_tokens = normalize_text(pred).split()
    r_tokens = normalize_text(ref).split()
    if not p_tokens or not r_tokens:
        return 0.0

    p_count = {}
    for t in p_tokens:
        p_count[t] = p_count.get(t, 0) + 1

    r_count = {}
    for t in r_tokens:
        r_count[t] = r_count.get(t, 0) + 1

    overlap = 0
    for t, c in p_count.items():
        overlap += min(c, r_count.get(t, 0))

    precision = overlap / len(p_tokens)
    recall = overlap / len(r_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

In [46]:
def citation_score(answer: str, expected_sig: dict) -> float:
    norm = normalize_text(answer)
    score = 0.0
    checks = 0

    if expected_sig["article_no"]:
        checks += 1
        if f"dieu {expected_sig['article_no']}" in norm:
            score += 1

    if expected_sig["year"]:
        checks += 1
        if expected_sig["year"] in norm:
            score += 1

    if expected_sig["doc_type"]:
        checks += 1
        doc_map = {"BOLUAT": "bo luat", "LUAT": "luat", "ND": "nghi dinh"}
        if doc_map.get(expected_sig["doc_type"], "") in norm:
            score += 1

    if checks == 0:
        return 0.0
    return score / checks

In [47]:
def build_context_text(hits: list[dict]) -> str:
    blocks = []
    for h in hits:
        cite = f"[{h['doc_type']} {h['year']} Điều {h['article_no']}]"
        blocks.append(f"{cite}\n{h['content']}")
    return "\n\n".join(blocks)

In [48]:
def groundedness_score(answer: str, hits: list[dict]) -> float:
    ans = normalize_text(answer)
    if not ans:
        return 0.0

    context = normalize_text(" ".join(h.get("content", "") for h in hits))
    if not context:
        return 0.0

    a_tokens = ans.split()
    if not a_tokens:
        return 0.0

    covered = sum(1 for t in a_tokens if t in context)
    return covered / len(a_tokens)

In [49]:
# Chuẩn bị bộ câu hỏi đánh giá
questions_df = pd.read_csv(EVAL_PATH).dropna(subset=["question", "expected_article", "expected_answer"])

# Loại các dòng header bị lặp trong CSV (ví dụ id='id')
questions_df["id"] = pd.to_numeric(questions_df["id"], errors="coerce")
questions_df = questions_df.dropna(subset=["id"])
questions_df["id"] = questions_df["id"].astype(int)

questions_df = questions_df.drop_duplicates(subset=["id", "question", "expected_article"])

print("Số câu hỏi dùng để đánh giá:", len(questions_df))
questions_df.head(5)

Số câu hỏi dùng để đánh giá: 100


,id,question,expected_answer,expected_article,topic
0,1,Độ tuổi lao động tối thiểu của người lao động ...,Độ tuổi lao động tối thiểu của người lao động ...,Điều 3 Bộ luật Lao động 2019,Lao động
1,2,Thời gian thử việc đối với công việc yêu cầu t...,Thời gian thử việc không quá 60 ngày đối với c...,Điều 25 Bộ luật Lao động 2019,Lao động
2,3,"Trong Bộ luật Dân sự, bất động sản bao gồm nhữ...","Bất động sản bao gồm: Đất đai; Nhà, công trình...",Điều 107 Bộ luật Dân sự 2015,Dân sự
3,4,Hoa lợi được định nghĩa như thế nào trong Bộ l...,Hoa lợi là sản vật tự nhiên mà tài sản mang lạ...,Điều 109 Bộ luật Dân sự 2015,Dân sự
4,5,Đất đai tại Việt Nam thuộc sở hữu của ai?,Đất đai thuộc sở hữu toàn dân do Nhà nước đại ...,Điều 12 Luật Đất đai 2024,Đất đai


In [50]:
# 1) RETRIEVAL EVALUATION - Khởi tạo
retrieval_rows = []

In [51]:
# 1) RETRIEVAL EVALUATION - Chạy retrieval từng câu
for _, row in questions_df.iterrows():
    qid = int(row["id"])
    expected_sig = extract_expected_signature(row["expected_article"])
    hits = retrieve(row["question"], top_k=TOP_K)
    rank = first_correct_rank(expected_sig, hits)

    retrieval_rows.append({
        "id": qid,
        "topic": row.get("topic", ""),
        "question": row["question"],
        "expected_article": row["expected_article"],
        "correct_rank": rank,
        f"hit@{TOP_K}": 1 if rank is not None else 0,
        "rr": 1.0 / rank if rank else 0.0,
        "top1_id": hits[0]["id"] if hits else "",
        "top1_article": f"Điều {hits[0]['article_no']} - {hits[0]['doc_type']} {hits[0]['year']}" if hits else "",
    })

print("Đã xử lý xong retrieval cho", len(retrieval_rows), "câu")

Đã xử lý xong retrieval cho 100 câu


In [52]:
# 1) RETRIEVAL EVALUATION - Tổng hợp DataFrame + Summary
retrieval_df = pd.DataFrame(retrieval_rows)
retrieval_summary = {
    "n_questions": int(len(retrieval_df)),
    f"hit@{TOP_K}": float(retrieval_df[f"hit@{TOP_K}"].mean()),
    "mrr": float(retrieval_df["rr"].mean()),
}
retrieval_summary

{'n_questions': 100, 'hit@5': 0.91, 'mrr': 0.7591666666666668}

In [53]:
# 1) RETRIEVAL EVALUATION - Hiển thị
print("=== Retrieval Summary ===")
for k, v in retrieval_summary.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

retrieval_df.head(10)

=== Retrieval Summary ===
n_questions: 100
hit@5: 0.9100
mrr: 0.7592


,id,topic,question,expected_article,correct_rank,hit@5,rr,top1_id,top1_article
0,1,Lao động,Độ tuổi lao động tối thiểu của người lao động ...,Điều 3 Bộ luật Lao động 2019,4.0,1,0.250000,ND_12_2022_XuPhatLaoDong|dieu_29|c2,Điều 29 - ND 2022
1,2,Lao động,Thời gian thử việc đối với công việc yêu cầu t...,Điều 25 Bộ luật Lao động 2019,NaN,0,0.000000,LUAT_124_2025_GiaoDucNgheNghiep|dieu_33|c2,Điều 33 - LUAT 2025
2,3,Dân sự,"Trong Bộ luật Dân sự, bất động sản bao gồm nhữ...",Điều 107 Bộ luật Dân sự 2015,3.0,1,0.333333,LUAT_31_2024_DatDai|dieu_236|c2,Điều 236 - LUAT 2024
3,4,Dân sự,Hoa lợi được định nghĩa như thế nào trong Bộ l...,Điều 109 Bộ luật Dân sự 2015,1.0,1,1.000000,BOLUAT_91_2015_DANSU|dieu_86|c1,Điều 86 - BOLUAT 2015
4,5,Đất đai,Đất đai tại Việt Nam thuộc sở hữu của ai?,Điều 12 Luật Đất đai 2024,1.0,1,1.000000,LUAT_31_2024_DatDai|dieu_44|c1,Điều 44 - LUAT 2024
5,6,Đất đai,Công dân có quyền tiếp cận những thông tin đất...,Điều 24 Luật Đất đai 2024,1.0,1,1.000000,LUAT_31_2024_DatDai|dieu_18|c1,Điều 18 - LUAT 2024
6,7,Kinh doanh Bất động sản,Các loại bất động sản nào được đưa vào kinh do...,Điều 5 Luật Kinh doanh bất động sản 2023,1.0,1,1.000000,LUAT_29_2023_KinhDoanhBatDongSan|dieu_44|c3,Điều 44 - LUAT 2023
7,8,Xử phạt Xây dựng,Hành vi thu tiền của bên mua nhà ở hình thành ...,Điều 58 Nghị định 16/2022/NĐ-CP,3.0,1,0.333333,ND_123_2024_XuPhatDatDai|dieu_5|c2,Điều 5 - ND 2024
8,9,Xử phạt Giao thông,Mức phạt đối với hành vi điều khiển xe ô tô ch...,Điều 6 Nghị định 168/2024/NĐ-CP,1.0,1,1.000000,ND_168_2024_XuPhatGiaoThong|dieu_7|c7,Điều 7 - ND 2024
9,10,Xử phạt Giao thông,Người điều khiển xe máy đi vào đường cao tốc b...,Điều 7 Nghị định 168/2024/NĐ-CP,1.0,1,1.000000,ND_168_2024_XuPhatGiaoThong|dieu_8|c1,Điều 8 - ND 2024


In [54]:
def generate_answer(question: str, hits: list[dict]) -> str:
    if client is None:
        return "SKIPPED_NO_API_KEY"

    context_text = build_context_text(hits)
    prompt = f"""
Bạn là trợ lý pháp luật Việt Nam.
Chỉ dựa trên ngữ cảnh được cung cấp để trả lời.
Nếu không đủ căn cứ, hãy nói rõ không đủ căn cứ.

Ngữ cảnh:
{context_text}

Câu hỏi:
{question}
"""

    try:
        resp = client.chat.completions.create(
            model=OPENROUTER_MODEL,
            messages=[
                {"role": "system", "content": "Bạn là trợ lý pháp luật Việt Nam."},
                {"role": "user", "content": prompt},
            ],
            temperature=0.2,
        )
        return resp.choices[0].message.content.strip()
    except Exception as exc:
        return f"ERROR: {exc}"

In [55]:
client = None
if OPEN_ROUTER_API_KEY:
    client = OpenAI(api_key=OPEN_ROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")

In [56]:
# 2) GENERATION + 3) END-TO-END - Khởi tạo
gen_rows = []

In [57]:
# 2) GENERATION + 3) END-TO-END - Chạy từng câu
for _, row in questions_df.iterrows():
    qid = int(row["id"])
    expected_sig = extract_expected_signature(row["expected_article"])
    hits = retrieve(row["question"], top_k=TOP_K)
    answer = generate_answer(row["question"], hits)

    f1 = 0.0 if answer.startswith("SKIPPED") or answer.startswith("ERROR:") else token_f1(answer, row["expected_answer"])
    c_score = 0.0 if answer.startswith("SKIPPED") or answer.startswith("ERROR:") else citation_score(answer, expected_sig)
    g_score = 0.0 if answer.startswith("SKIPPED") or answer.startswith("ERROR:") else groundedness_score(answer, hits)

    rank = first_correct_rank(expected_sig, hits)
    r_score = 1.0 if rank is not None else 0.0

    e2e = 0.35 * r_score + 0.4 * f1 + 0.15 * c_score + 0.10 * g_score

    gen_rows.append({
        "id": qid,
        "topic": row.get("topic", ""),
        "question": row["question"],
        "expected_article": row["expected_article"],
        "expected_answer": row["expected_answer"],
        "generated_answer": answer,
        "retrieval_hit": r_score,
        "generation_f1": f1,
        "citation_score": c_score,
        "groundedness": g_score,
        "end2end_score": e2e,
    })

print("Đã xử lý generation/end-to-end cho", len(gen_rows), "câu")

Đã xử lý generation/end-to-end cho 100 câu


In [58]:
# 2) GENERATION + 3) END-TO-END - Tạo DataFrame kết quả
result_df = pd.DataFrame(gen_rows)
result_df.shape

(100, 11)

In [59]:
# 2) GENERATION SUMMARY
print("=== Generation Summary ===")
print("generation_f1:", round(float(result_df["generation_f1"].mean()), 4))
print("citation_score:", round(float(result_df["citation_score"].mean()), 4))
print("groundedness:", round(float(result_df["groundedness"].mean()), 4))

=== Generation Summary ===
generation_f1: 0.4827
citation_score: 0.5
groundedness: 0.9324


In [60]:
# 3) END-TO-END SUMMARY
print("=== End-to-End Summary ===")
print("end2end_score:", round(float(result_df["end2end_score"].mean()), 4))

if (result_df["generated_answer"] == "SKIPPED_NO_API_KEY").all():
    print("⚠️ Chưa có OPEN_ROUTER_API_KEY nên phần Generation/End-to-End chưa gọi được LLM.")

=== End-to-End Summary ===
end2end_score: 0.6798


In [61]:
# Xem nhanh kết quả mẫu
result_df.head(10)

,id,topic,question,expected_article,expected_answer,generated_answer,retrieval_hit,generation_f1,citation_score,groundedness,end2end_score
0,1,Lao động,Độ tuổi lao động tối thiểu của người lao động ...,Điều 3 Bộ luật Lao động 2019,Độ tuổi lao động tối thiểu của người lao động ...,Độ tuổi lao động tối thiểu của người lao động ...,1.0,0.280488,0.5,0.948148,0.632010
1,2,Lao động,Thời gian thử việc đối với công việc yêu cầu t...,Điều 25 Bộ luật Lao động 2019,Thời gian thử việc không quá 60 ngày đối với c...,Không đủ căn cứ. Ngữ cảnh cung cấp không đề cậ...,0.0,0.622951,0.0,0.967742,0.345955
2,3,Dân sự,"Trong Bộ luật Dân sự, bất động sản bao gồm nhữ...",Điều 107 Bộ luật Dân sự 2015,"Bất động sản bao gồm: Đất đai; Nhà, công trình...","Trong Bộ luật Dân sự, bất động sản bao gồm:\n\...",1.0,0.761905,1.0,0.953125,0.900074
3,4,Dân sự,Hoa lợi được định nghĩa như thế nào trong Bộ l...,Điều 109 Bộ luật Dân sự 2015,Hoa lợi là sản vật tự nhiên mà tài sản mang lạ...,Không đủ căn cứ. Ngữ cảnh được cung cấp không ...,1.0,0.111111,0.5,0.826087,0.552053
4,5,Đất đai,Đất đai tại Việt Nam thuộc sở hữu của ai?,Điều 12 Luật Đất đai 2024,Đất đai thuộc sở hữu toàn dân do Nhà nước đại ...,Không đủ căn cứ để xác định cụ thể ai là người...,1.0,0.346939,0.5,0.870130,0.650788
5,6,Đất đai,Công dân có quyền tiếp cận những thông tin đất...,Điều 24 Luật Đất đai 2024,"Công dân được tiếp cận quy hoạch, kế hoạch sử ...",Công dân có quyền tiếp cận các thông tin đất đ...,1.0,0.401606,1.0,0.989950,0.759638
6,7,Kinh doanh Bất động sản,Các loại bất động sản nào được đưa vào kinh do...,Điều 5 Luật Kinh doanh bất động sản 2023,Bao gồm: Nhà ở có sẵn và nhà ở hình thành tron...,"Theo Luật Kinh doanh bất động sản, các loại bấ...",1.0,0.322275,0.5,0.962963,0.650206
7,8,Xử phạt Xây dựng,Hành vi thu tiền của bên mua nhà ở hình thành ...,Điều 58 Nghị định 16/2022/NĐ-CP,Phạt tiền từ 100.000.000 đồng đến 120.000.000 ...,Không đủ căn cứ để xác định mức phạt cụ thể ch...,1.0,0.365385,0.0,0.921569,0.588311
8,9,Xử phạt Giao thông,Mức phạt đối với hành vi điều khiển xe ô tô ch...,Điều 6 Nghị định 168/2024/NĐ-CP,Phạt tiền từ 12.000.000 đồng đến 14.000.000 đồ...,Không đủ căn cứ để xác định mức phạt đối với h...,1.0,0.323810,0.0,0.947368,0.574261
9,10,Xử phạt Giao thông,Người điều khiển xe máy đi vào đường cao tốc b...,Điều 7 Nghị định 168/2024/NĐ-CP,Phạt tiền từ 4.000.000 đồng đến 6.000.000 đồng...,Người điều khiển xe máy đi vào đường cao tốc s...,1.0,0.440678,1.0,0.951220,0.771393


In [62]:
# Xuất báo cáo
REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(exist_ok=True)

retrieval_out = REPORT_DIR / "retrieval_eval.csv"
generation_out = REPORT_DIR / "generation_end2end_eval.csv"
summary_out = REPORT_DIR / "summary_eval.json"

retrieval_df.to_csv(retrieval_out, index=False, encoding="utf-8-sig")
result_df.to_csv(generation_out, index=False, encoding="utf-8-sig")

summary = {
    "retrieval": retrieval_summary,
    "generation": {
        "generation_f1": float(result_df["generation_f1"].mean()),
        "citation_score": float(result_df["citation_score"].mean()),
        "groundedness": float(result_df["groundedness"].mean()),
    },
    "end_to_end": {
        "end2end_score": float(result_df["end2end_score"].mean())
    },
    "n_questions": int(len(result_df)),
}

with summary_out.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved:")
print("-", retrieval_out)
print("-", generation_out)
print("-", summary_out)
summary

Saved:
- d:\sourcecode\4th year\2nd term\CĐHTTT\BTLCuoiKy\reports\retrieval_eval.csv
- d:\sourcecode\4th year\2nd term\CĐHTTT\BTLCuoiKy\reports\generation_end2end_eval.csv
- d:\sourcecode\4th year\2nd term\CĐHTTT\BTLCuoiKy\reports\summary_eval.json


{'retrieval': {'n_questions': 100, 'hit@5': 0.91, 'mrr': 0.7591666666666668},
 'generation': {'generation_f1': 0.4827356285955964,
  'citation_score': 0.5,
  'groundedness': 0.9324477072516923},
 'end_to_end': {'end2end_score': 0.6798390221634076},
 'n_questions': 100}